In [ ]:
!pip3 install google-genai chromadb python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found

# Load the packages


In [ ]:
import json
import os
import chromadb
from dotenv import load_dotenv
from google import genai
from google.genai import types

In [ ]:
from google.colab import userdata
gemini_api_key=userdata.get('GEMINI')

In [ ]:
client = genai.Client(api_key=gemini_api_key)
print("Gemini client created successfully!!")

Gemini client created successfully!!


In [ ]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="What is RAG? Explain with 3 bullet points.",
    config=types.GenerateContentConfig(
        system_instruction="You are a helpful assistant."
    ),
)

print(response.text)

**RAG (Retrieval-Augmented Generation)** is an AI framework designed to make Large Language Models (LLMs) more accurate and reliable. Here is how it works:

* **Definition & Concept:** RAG connects an AI text generator to external knowledge sources (like company databases, internal documents, or the live web), allowing the AI to look up facts before answering a prompt.
* **How It Works:** It operates in two main steps: first, it **retrieves** relevant information from a specific dataset based on the user's question; second, it uses that context to **generate** a precise, informed response.
* **Key Benefits:** It prevents the AI from making things up (hallucinating), allows access to real-time or private data, and saves money by avoiding the need to constantly retrain expensive AI models.


In [ ]:
with open("/content/company_hr_policy.txt", "r") as f:
  hr_document = f.read()

with open("/content/engineering_standards.txt", "r") as f:
  engineering_document = f.read()

with open("/content/onboarding_guide.txt", "r") as f:
  onboarding = f.read()

with open("/content/product_knowledge_base.txt", "r") as f:
  prodcut = f.read()

with open("/content/security_policy.txt", "r") as f:
  security = f.read()
print(engineering_document)

NovaTech Solutions — Engineering Standards & Practices
Version 2.0 | Last Updated: March 2026

SECTION 1: GIT WORKFLOW

Branching Strategy:
We follow a simplified Git Flow. The main branch is always deployable. All development happens in feature branches.

Branch naming convention:
- feature/JIRA-123-short-description (for new features)
- fix/JIRA-456-bug-description (for bug fixes)
- hotfix/JIRA-789-critical-fix (for production emergencies)

Pull Request Rules:
Every change must go through a Pull Request (PR). Direct commits to main are blocked. PRs require at least 1 approval from a team member. The PR author cannot approve their own PR. All CI checks must pass before merging. PRs should be small — ideally under 400 lines of code changes.

Commit Message Format:
Use conventional commits format:
- feat: add user authentication endpoint
- fix: resolve null pointer in payment service
- docs: update API documentation for v2
- refactor: simplify database connection logic
- test: add unit 

In [ ]:
print(f"HR Policy Document : {len(hr_document)}")
print(f"Engineering Document : {len(engineering_document)}")

print(f"HR Policy Words: {len(hr_document.split())}")
print(f"Engineering Words: {len(engineering_document.split())}")

print(f"HR Policy Words: {len(hr_document.split())}")
print(f"Engineering Words: {len(engineering_document.split())}")

print(f"HR Policy Words: {len(hr_document.split())}")
print(f"Engineering Words: {len(engineering_document.split())}")

print(f"HR Policy Words: {len(hr_document.split())}")
print(f"Engineering Words: {len(engineering_document.split())}")

HR Policy Document : 7464
Engineering Document : 4941
HR Policy Words: 1052
Engineering Words: 680
HR Policy Words: 1052
Engineering Words: 680
HR Policy Words: 1052
Engineering Words: 680
HR Policy Words: 1052
Engineering Words: 680


#Chunking Strategy

In [ ]:
def chunk_document(text, source_name):
    """
    Split a document into chunks by paragraph.
    Returns a list of dicts with text and metadata.
    """
    # Split on double newlines
    paragraphs = text.strip().split("\n\n")

    chunks = []
    for para in paragraphs:
        para = para.strip()
        # Skip short lines (headers, separators)
        if len(para) < 50:
            continue
        # Skip separator lines
        if para.startswith("===="):
            continue
        chunks.append({
            "text": para,
            "source": source_name
        })

    return chunks

In [ ]:
hr_chunks = chunk_document(hr_document, "HR Policy")
engineering_chunks = chunk_document(engineering_document, "Engineering")
onboarding_chunks = chunk_document(onboarding, "Onboarding")
product_chunks = chunk_document(prodcut, "Product")
security_chunks = chunk_document(security, "Security")

In [ ]:
print(len(hr_chunks))
print(len(engineering_chunks))

25
18


In [ ]:
all_chunks = (
    hr_chunks
    + engineering_chunks
    + onboarding_chunks
    + product_chunks
    + security_chunks
)
print(f"Total chunks created: {len(all_chunks)}")


Total chunks created: 120


#Embedding Mnaually

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences = ''' Office Network:
- Connect only to the "NovaTech-Secure" WiFi network (WPA3)
- The "NovaTech-Guest" network is for visitors only — employees should not use it
- Do not connect personal IoT devices (smart speakers, etc.) to the office network '''

embeddings = model.encode(sentences)

print(embeddings.shape)
print(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(384,)
[-2.71274522e-02  3.71070579e-02  2.69963648e-02 -3.76516432e-02
 -1.24880141e-02  3.69489342e-02  2.72595212e-02 -9.89013687e-02
  1.21910451e-02 -5.96426195e-03 -2.40616184e-02  9.83860716e-02
 -1.46336211e-02 -7.14525115e-03 -9.63099953e-03 -3.73131176e-03
  4.77994904e-02 -6.60055950e-02  8.68471265e-02  5.59232198e-02
  6.26666099e-02 -2.16602795e-02  5.59136644e-03 -4.52482216e-02
  1.12407371e-01  1.82983223e-02  1.06359925e-02  8.20038021e-02
 -1.59085672e-02  1.81507748e-02 -2.22688392e-02  1.61229190e-03
  2.39930465e-04  4.42378670e-02 -4.55207527e-02 -1.53136328e-01
 -4.13969159e-03  6.60401806e-02 -2.05189046e-02 -1.09434910e-02
 -1.74337719e-02 -5.66976145e-02 -7.42021725e-02  4.75947820e-02
  9.90143139e-03 -3.26359086e-02 -6.63817152e-02 -5.27318902e-02
  1.69445630e-02  1.43104633e-02  1.61807705e-02 -7.82664269e-02
 -6.00268357e-02  1.23774938e-01  2.48117596e-02 -8.87978747e-02
  8.00592452e-03  2.06095278e-02  3.16744410e-02 -4.53798324e-02
  7.19641522e-02 -

In [ ]:
sentence1 = ''' I love chicken Biriyani '''
sentence2 = ''' I love eating flaovured rice with chicken'''
sentence3 = ''' I am doing work from Home'''

embedding1 = model.encode(sentence1)
embedding2 = model.encode(sentence2)
embedding3 = model.encode(sentence3)

print(embedding1.shape)
print(embedding2.shape)
print(embedding3.shape)

(384,)
(384,)
(384,)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarity1 = cosine_similarity([embedding1], [embedding2])
similarity2 = cosine_similarity([embedding1], [embedding3])
similarity3 = cosine_similarity([embedding2], [embedding3])

print(similarity1)
print(similarity2)
print(similarity3)

[[0.56787026]]
[[0.01995531]]
[[0.10406436]]


#Storing chunks in Chroma DB

In [ ]:
chroma_clinet = chromadb.Client()
collection = chroma_clinet.create_collection(name="company_docs")

In [ ]:
documents = []
ids = []
metadatas = []

for i, chunk in enumerate(all_chunks):
  documents.append(chunk['text'])
  ids.append(f"chunk_{i}")
  metadatas.append({"source": chunk["source"]})

print(documents[110])
print(ids[110])
print(metadatas[110])

Office Network:
- Connect only to the "NovaTech-Secure" WiFi network (WPA3)
- The "NovaTech-Guest" network is for visitors only — employees should not use it
- Do not connect personal IoT devices (smart speakers, etc.) to the office network
chunk_110
{'source': 'Security'}


In [ ]:
collection.add(
    documents = documents,
    ids = ids,
    metadatas = metadatas
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 98.3MiB/s]


In [ ]:
print(f" Stored {len(documents)} chunks in chroma db")
print(f" Sources: 5 differnt files")

 Stored 120 chunks in chroma db
 Sources: 5 differnt files


#Building Retrievel Pipeline

In [ ]:
def retrieve(question, n_results = 3):
  results = collection.query(
      query_texts = [question],
      n_results = n_results
  )
  return results['documents'][0], results['metadatas'][0]

In [ ]:
chunks, sources = retrieve("What is the work from home policy?", 3)

for i in range(len(chunks)):
  print(f"--- Chunks {i+1} ---")
  print(f"Source: {sources[i]}")
  print(f"Text: {chunks[i]} ")
  print()

--- Chunks 1 ---
Source: {'source': 'HR Policy'}
Text: Eligibility:
All employees who have completed their probation period (6 months) are eligible for Work From Home (WFH) arrangements. Employees in their probation period may request WFH only in exceptional circumstances with manager and HR approval. 

--- Chunks 2 ---
Source: {'source': 'HR Policy'}
Text: NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: January 2026 

--- Chunks 3 ---
Source: {'source': 'HR Policy'}
Text: Regular WFH:
Employees may work from home up to 2 days per week. The preferred WFH days are Wednesday and Friday, though teams may adjust based on project needs. Employees must be available during core working hours (10:00 AM to 6:00 PM IST) on WFH days. 



In [ ]:
def ask_rag(question, n_results=3, verbose=True):
    chunks, sources = retrieve(question, n_results)

    if verbose:
        print(f"\n{'═' * 60}")
        print(f"❓ Question: {question}")
        print(f"{'─' * 60}")
        print(f"📄 Retrieved {len(chunks)} chunks:")
        for chunk, source in zip(chunks, sources):
            print(f"   [{source['source']}] {chunk[:80]}...")
        print(f"{'─' * 60}")

    context = "\n\n".join(chunks)

    system_prompt = (
        "You are a helpful assistant that answers questions based ONLY on the provided context.\n"
        "If the context does not contain information to answer this question, "
        "say 'I don't have enough context or information to answer this question.'\n"
        "Do not make up information or assume anything. Strictly answer only from the provided context."
    )

    user_prompt = f"Context:\n{context}\n\n---\n\nQuestion: {question}"

    # Call Gemini API
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=0.2,  # Low temperature for factual grounding
        ),
    )

    answer = response.text

    if verbose:
        print(f"💡 Answer: {answer}")
        print(f"{'═' * 60}")

    return answer


print("RAG Pipeline is built")

RAG Pipeline is built


#
# Test the RAG Pipeline

In [ ]:
ask_rag("What is Home Policy?")


════════════════════════════════════════════════════════════
❓ Question: What is Home Policy?
────────────────────────────────────────────────────────────
📄 Retrieved 3 chunks:
   [HR Policy] Eligibility:
All employees who have completed their probation period (6 months) ...
   [Security] NovaTech Solutions — Information Security Policy
Classification: Internal | Vers...
   [Security] Company Laptops:
- Full disk encryption must be enabled (FileVault on Mac, BitLo...
────────────────────────────────────────────────────────────
💡 Answer: I don't have enough context or information to answer this question.
════════════════════════════════════════════════════════════


"I don't have enough context or information to answer this question."

In [ ]:
ask_rag("What is the work from Home Policy?")


════════════════════════════════════════════════════════════
❓ Question: What is the work from Home Policy?
────────────────────────────────────────────────────────────
📄 Retrieved 3 chunks:
   [HR Policy] Eligibility:
All employees who have completed their probation period (6 months) ...
   [HR Policy] NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: J...
   [HR Policy] Regular WFH:
Employees may work from home up to 2 days per week. The preferred W...
────────────────────────────────────────────────────────────
💡 Answer: Based on the provided context, the Work From Home (WFH) Policy includes the following rules:

* **Eligibility:** Employees are eligible for WFH after completing their 6-month probation period. Employees currently in probation may only request WFH in exceptional circumstances with manager and HR approval.
* **Regular WFH Allowance:** Employees may work from home up to 2 days per week.
* **Preferred Days:** The preferred WFH days are Wedn

'Based on the provided context, the Work From Home (WFH) Policy includes the following rules:\n\n* **Eligibility:** Employees are eligible for WFH after completing their 6-month probation period. Employees currently in probation may only request WFH in exceptional circumstances with manager and HR approval.\n* **Regular WFH Allowance:** Employees may work from home up to 2 days per week.\n* **Preferred Days:** The preferred WFH days are Wednesday and Friday, but teams can adjust these based on project needs.\n* **Core Hours:** Employees must be available during core working hours from 10:00 AM to 6:00 PM IST on WFH days.'

In [ ]:
# Product KB questions
ask_rag("What are the pricing plans for CloudDesk Pro?", 10)


════════════════════════════════════════════════════════════
❓ Question: What are the pricing plans for CloudDesk Pro?
────────────────────────────────────────────────────────────
📄 Retrieved 10 chunks:
   [Product] Support Channels:
- Help Center: docs.clouddesk.pro — Self-service articles and ...
   [Product] What is CloudDesk Pro?
CloudDesk Pro is a cloud-based project management and tea...
   [Product] CloudDesk Pro — Product Knowledge Base
Internal Support Reference | Version 2.1 ...
   [Product] Target Users:
CloudDesk Pro is designed for project managers, team leads, and in...
   [Product] Enterprise Plan — Custom pricing (contact sales):
- Everything in Business, plus...
   [Product] Starter Plan — Rs 299/user/month (billed annually) or Rs 399/user/month (billed ...
   [Product] Business Plan — Rs 699/user/month (billed annually) or Rs 899/user/month (billed...
   [Product] Annual Billing Discount:
All plans offer approximately 25% discount when billed ...
   [Product] Complia

'Based on the provided context, the pricing plans for CloudDesk Pro are:\n\n* **Starter Plan:** Rs 299/user/month (billed annually) or Rs 399/user/month (billed monthly)\n* **Business Plan:** Rs 699/user/month (billed annually) or Rs 899/user/month (billed monthly)\n* **Enterprise Plan:** Custom pricing (contact sales)'

In [ ]:
# A question that spans both documents — should say "not enough info"
ask_rag("What is the capital of France?")


════════════════════════════════════════════════════════════
❓ Question: What is the capital of France?
────────────────────────────────────────────────────────────
📄 Retrieved 3 chunks:
   [Product] What is CloudDesk Pro?
CloudDesk Pro is a cloud-based project management and tea...
   [Onboarding] Communication:
- Slack: Primary daily communication. Check channels: #general, #...
   [HR Policy] Communication:
Slack is the primary communication tool for daily work. Email is ...
────────────────────────────────────────────────────────────
💡 Answer: I don't have enough context or information to answer this question.
════════════════════════════════════════════════════════════


"I don't have enough context or information to answer this question."

#Agentic RAG

#Create RAG Tool

In [ ]:
def search_docs(query:str) ->str:
  results = collection.query(
      query_texts = [query],
      n_results = 3
  )
  chunks = results['documents'][0]
  return "\n\n".join(chunks)

#Json Schema

rag_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_docs",
            "description": "Search the company internal documents (HR Policy, Product, Security, onboarding, engineering) to find the relevant information. Use this when the user asks about company policies, products, features or support.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query you want to find relevant document"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

available_tools = {
    "search_docs": search_docs
}
print("Rag tool defined")

Rag tool defined


In [ ]:
def rag_agent(question: str, verbose: bool = True):
    if verbose:
        print(f"\n{'═' * 60}")
        print(f"🧑 Question: {question}")
        print(f"{'─' * 60}")

    system_instruction = (
        "You are a helpful company assistant with access to internal documents "
        "including HR policies, Product, Engineering, Onboarding, and Security.\n"
        "Use the search_docs tool to find answers from company documents.\n"
        "If the documents do not contain the answer, say so clearly.\n"
        "Always base your answers on the retrieved documents if you decide to use the tool.\n"
        "If tool use is not needed, specify that you are not using the tool and answer based on your knowledge."
    )

    chat = client.chats.create(
        model="gemini-3.6-flash",
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            tools=[search_docs],
            temperature=0.2,
        ),
    )

    response = chat.send_message(question)
    answer = response.text

    if verbose:
        print(f"🤖 Answer: {answer}")
        print(f"{'═' * 60}")

    return answer


print("✅ RAG Agent ready!")

✅ RAG Agent ready!


In [ ]:

rag_agent("How many sick leave days do I get per year?")


════════════════════════════════════════════════════════════
🧑 Question: How many sick leave days do I get per year?
────────────────────────────────────────────────────────────
🤖 Answer: Based on the company's HR policy, full-time employees are entitled to **12 days of sick leave per year**.

Key details regarding sick leave:
* **Medical Certificate:** Required if sick leave is taken for more than 3 consecutive days.
* **Carry Forward / Encasement:** Sick leave cannot be carried forward to the next year or encashed.
* **Extended Illness:** If you need more than 12 days due to an extended illness, you may apply for medical leave without pay (subject to HR approval).
════════════════════════════════════════════════════════════


"Based on the company's HR policy, full-time employees are entitled to **12 days of sick leave per year**.\n\nKey details regarding sick leave:\n* **Medical Certificate:** Required if sick leave is taken for more than 3 consecutive days.\n* **Carry Forward / Encasement:** Sick leave cannot be carried forward to the next year or encashed.\n* **Extended Illness:** If you need more than 12 days due to an extended illness, you may apply for medical leave without pay (subject to HR approval)."

In [ ]:
rag_agent("What is 25*4 ?")



════════════════════════════════════════════════════════════
🧑 Question: What is 25*4 ?
────────────────────────────────────────────────────────────
🤖 Answer: I am not using the search tool for this question.

25 * 4 = 100
════════════════════════════════════════════════════════════


'I am not using the search tool for this question.\n\n25 * 4 = 100'

In [ ]:
rag_agent("What is the capital of france?")


════════════════════════════════════════════════════════════
🧑 Question: What is the capital of france?
────────────────────────────────────────────────────────────
🤖 Answer: I am not using the search tool for this question as it is a matter of general knowledge.

The capital of France is Paris.
════════════════════════════════════════════════════════════


'I am not using the search tool for this question as it is a matter of general knowledge.\n\nThe capital of France is Paris.'